# Ch 22 — 기성 sLLM 고르고 쓰기

HuggingFace Hub 의 sLLM 5종(Phi-3 / SmolLM2 / Gemma 2 / Qwen 2.5 / Llama 3.2)을 비교하고, 본인 도메인에 맞는 모델을 결정 트리로 선택하는 챕터.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/desty/study-tiny-llm/blob/main/notebooks/part7/ch22_choosing_slm.ipynb)

In [ ]:
# !pip install -q transformers huggingface_hub accelerate

import torch
import json
from huggingface_hub import HfApi, hf_hub_download

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. 모델 카드 30초 평가 (최소 예제)

HuggingFace Hub API 로 모델 7항목을 빠르게 조회하는 최소 예제.

In [ ]:
# model_info.py
from huggingface_hub import HfApi, hf_hub_download
import json

def model_summary(repo_id):
    api = HfApi()
    info = api.model_info(repo_id)
    cfg = json.load(open(hf_hub_download(repo_id, "config.json")))
    return {
        "repo_id": repo_id,
        "params":  cfg.get("num_parameters") or "?",
        "context": cfg.get("max_position_embeddings", "?"),
        "vocab":   cfg.get("vocab_size", "?"),
        "license": info.cardData.get("license", "?") if info.cardData else "?",
        "downloads": info.downloads, "likes": info.likes,
    }

# 예시: SmolLM2-1.7B
summary = model_summary("HuggingFaceTB/SmolLM2-1.7B-Instruct")
for k, v in summary.items():
    print(f"  {k}: {v}")

## 4. 한국어 능력 실측 (실전)

5종 모델에 한국어 prompt 4개를 던져 답변 품질을 비교하는 실전 코드.

> **주의**: 5개 모델을 순차 로드하므로 RAM/VRAM 16 GB+ 권장. 메모리 부족 시 모델 1~2개씩 확인.

In [ ]:
# ko_compare.py
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

prompts = ["안녕하세요. 자기소개를 해주세요.",
           "다음을 한 줄 요약: 인공지능은 ...",
           "Python으로 fibonacci 함수.",
           "오늘 점심 추천."]

# 메모리 절약: 하나씩 로드 후 삭제
# 라이선스 동의 필요 모델은 주석 처리
MODELS = [
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    # "google/gemma-2-2b-it",                   # Gemma 라이선스 동의 필요
    # "meta-llama/Llama-3.2-1B-Instruct",       # Llama 라이선스 동의 필요
    # "microsoft/Phi-3.5-mini-instruct",         # 3.8B, 메모리 주의
]

for m in MODELS:
    print(f"\n=== {m} ===")
    tok = AutoTokenizer.from_pretrained(m)
    model = AutoModelForCausalLM.from_pretrained(m, torch_dtype=torch.bfloat16, device_map="auto")
    for p in prompts:
        msgs = [{"role": "user", "content": p}]
        ids = tok.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True)
        ids = ids.to(model.device)
        out = model.generate(ids, max_new_tokens=200, do_sample=False)
        ans = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        print(f"  Q: {p}")
        print(f"  A: {ans[:150]}\n")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## 5. 도구 호출 (function calling) 테스트

In [ ]:
# tool_call_test.py
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

m = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(m)
model = AutoModelForCausalLM.from_pretrained(m, torch_dtype=torch.bfloat16, device_map="auto")

TOOLS = [{"type": "function", "function": {
    "name": "get_weather", "description": "도시 날씨",
    "parameters": {"type": "object",
                   "properties": {"city": {"type": "string"}},
                   "required": ["city"]},
}}]
msgs = [{"role": "user", "content": "서울 날씨 어때?"}]
ids = tok.apply_chat_template(msgs, tools=TOOLS, return_tensors="pt", add_generation_prompt=True)
ids = ids.to(model.device)
out = model.generate(ids, max_new_tokens=200)
print(tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True))
# 기대 출력: {"name": "get_weather", "arguments": {"city": "서울"}}

## 7. 결정 트리

```
1. 한국어 주 도메인?  Yes → Qwen 2.5
2. 노트북 16GB 만?    Yes → 1.5B~3B / No → 7B+
3. 라이선스 엄격?     Yes → Phi-3 / Qwen 2.5
4. 도구 호출?         Yes → Qwen 2.5 / Llama 3.2 / Phi-3.5
5. 본인 LoRA 데이터 양?  >=10K → 1.5B~3B / <1K → 0.5B~1B
```

본 책 캡스톤 (한국 동화) 답: **Qwen 2.5-0.5B**.

## 연습문제

1. 본인 도메인 prompt 5개를 5 모델에 던져 표 정리.
2. `model_summary` 로 5 모델 7항목 비교.
3. 영어 코드 생성기라면 어느 모델? 결정 트리 통과.
4. 본인 회사 작업에 §7 결정 트리 적용.
5. **(생각해볼 것)** "Qwen 2.5 = 한국어 표준" 이 1년 후에도 그대로일까?

In [ ]:
# 연습 1: 본인 도메인 prompt 5개 x 5 모델 비교


In [ ]:
# 연습 2: model_summary 로 5 모델 7항목 비교
models_to_check = [
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    # 추가 모델 넣기
]
for m in models_to_check:
    print(model_summary(m))


In [ ]:
# 연습 3~5: 결정 트리 적용 (텍스트 또는 코드로 답변)
